- 001 hello world
- 002 parallelize the main loop
- 003 joblib parallelization, should actually work this time
- 004 speed it up by removing cleaning, already done in encoding stage now (which itself should be separated out so this doesn't have to wait for encoding)


In [ ]:
# !pip install diff_patch_match

In [ ]:
from ppp_paths import dataint

import pandas as pd
import unicodedata
import re

# import json, requests 
# import networkx as nx
import gc

In [ ]:
possiblepairs = pd.read_parquet(dataint + 'potentialpairswithinwindow.parquet')
possiblepairs = possiblepairs[['magid', 'patent_id']]
possiblepairs['patent_id'] = possiblepairs['patent_id'].astype(str)
possiblepairs['magid'] = possiblepairs['magid'].astype(str)
# possiblepairs 

In [ ]:
patentcontent = pd.read_csv(dataint + "patenttoembed.tsv", sep='\t')
patentcontent.rename(columns={'id': 'patent_id', 'title': 'title_patent', 'abstract': 'abstract_patent'}, inplace=True)
patentcontent['patent_id'] = patentcontent['patent_id'].astype(str)
# patentcontent


In [ ]:
papercontent = pd.read_csv(dataint + "papertoembed.tsv", sep='\t')
papercontent.rename(columns={'id': 'magid', 'title': 'title_paper', 'abstract': 'abstract_paper'}, inplace=True)
papercontent['magid'] = papercontent['magid'].astype(str)
# papercontent

In [ ]:
possiblepairspat = pd.merge(possiblepairs, patentcontent, on='patent_id',  how='left')
possiblepairspatpaper = pd.merge(possiblepairspat, papercontent, on='magid',  how='left')
possiblepairspatpaper = possiblepairspatpaper[['magid', 'patent_id', 'title_patent', 'abstract_patent', 'title_paper', 'abstract_paper']]
possiblepairspatpaper = possiblepairspatpaper.dropna()
# possiblepairspatpaper

In [ ]:
# Step 1: Create concatenated and cleaned text columns
possiblepairspatpaper['text1'] = (possiblepairspatpaper['title_patent'] + ". " + possiblepairspatpaper['abstract_patent'])
#.apply(clean_string_stop_words)
possiblepairspatpaper['text2'] = (possiblepairspatpaper['title_paper'] + ". " + possiblepairspatpaper['abstract_paper'])
#.apply(clean_string_stop_words)
# print("apply")
possiblepairspatpaper = possiblepairspatpaper[['magid', 'patent_id', 'text1', 'text2']]


In [ ]:
for col in ['text1', 'text2']:
    possiblepairspatpaper[col] = possiblepairspatpaper[col].astype(str).str.slice(0, 1000)
possiblepairspatpaper['text1'].str.len().max()

In [ ]:
# possiblepairspatpaper['text1'] = possiblepairspatpaper['text1']
# possiblepairspatpaper['text2'] = possiblepairspatpaper['text2']
# del possiblepairspat 
# del possiblepairs 
# del papercontent 
# del patentcontent
# gc.collect()
possiblepairspatpaper.dropna()
possiblepairspatpaper.to_parquet(dataint + 'possiblepairspatpaper4plagarism.parquet', index=False, compression='snappy')

In [ ]:
# OPTIONALLY 
from ppp_paths import dataint

import pandas as pd

possiblepairspatpaper = pd.read_parquet(dataint + 'possiblepairspatpaper4plagarism.parquet')
possiblepairspatpaper = possiblepairspatpaper
#[:60000]
possiblepairspatpaper.shape

In [ ]:
# just do it for the actual PPPs in a post-hoc way
justforfinalppps = False
if justforfinalppps is True:
    datafin = './data/fin/' 
    finalppps = pd.read_csv(datafin + 'finalppps.csv')
    finalppps['magid'] = finalppps['magid'].astype(str)
    possiblepairspatpaper = pd.merge(possiblepairspatpaper, finalppps[['magid', 'patent_id']], on=['magid', 'patent_id'], how='inner')
    possiblepairspatpaper

In [ ]:
# possiblepairspatpaper = possiblepairspatpaper[:10000]


In [ ]:
import pandas as pd
import numpy as np
from joblib import Parallel, delayed
import multiprocessing
from tqdm import tqdm
import logging
import compute_plagiarism_metrics001  # Contains compute_plagiarism_metrics

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

# Function to process a single row
def process_row(row, func):
    try:
        result = func(row)
        return result
    except Exception as e:
        logging.error(f"Error processing row with text1: {row['text1'][:50]}...: {e}")
        return pd.Series()  # Return empty series for failed rows

# Parallel apply function using joblib
def parallel_apply(df, func, n_jobs):
    n_jobs = n_jobs or max(1, multiprocessing.cpu_count() // 2)
    logging.info(f"Using {n_jobs} workers")
    
    # Apply function to each row in parallel
    results = Parallel(n_jobs=n_jobs, verbose=10)(
        delayed(process_row)(row, func) for _, row in tqdm(df.iterrows(), total=len(df))
    )
    
    # Convert results to DataFrame
    results_df = pd.DataFrame([r for r in results if not r.empty])
    return results_df

if __name__ == '__main__':
    # Log dataset info
    logging.info(f"Dataset size: {len(possiblepairspatpaper)} rows")
    logging.info(f"Average text1 length: {possiblepairspatpaper['text1'].str.len().mean()}")
    logging.info(f"Average text2 length: {possiblepairspatpaper['text2'].str.len().mean()}")

    # Ensure text columns are strings
    possiblepairspatpaper['text1'] = possiblepairspatpaper['text1'].astype(str).fillna('')
    possiblepairspatpaper['text2'] = possiblepairspatpaper['text2'].astype(str).fillna('')

    # Apply plagiarism computation in parallel
    metrics = parallel_apply(possiblepairspatpaper, compute_plagiarism_metrics001.compute_plagiarism_metrics, n_jobs=24)
    # metrics = parallel_apply(possiblepairspatpaper, compute_plagiarism_metrics001.compute_plagiarism_metrics, batch_size=100, chunk_size=10, n_jobs=5)

    # Combine metrics with original DataFrame
    possiblepairspatpaper = pd.concat([possiblepairspatpaper, metrics], axis=1)

    # Save results
    # possiblepairspatpaper.to_csv('results.csv', index=False)
    # logging.info("Results saved to results.csv")

In [ ]:
metrics

In [ ]:
possiblepairspatpaper

In [ ]:
# Clean up temporary columns (optional)
possiblepairspatpaper = possiblepairspatpaper.drop(columns=['text1', 'text2'])
pd.set_option('display.max_colwidth', None)
possiblepairspatpaper

In [ ]:
# possiblepairspatpaper = possiblepairspatpaper[['magid', 'patent_id', 'plagiarism_percentage', 'plagiarism_group_words_1',
                                            #    'plagiarism_group_words_3', 'plagiarism_group_words_5']]
if justforfinalppps is True:
    possiblepairspatpaper.to_csv(dataint + 'finalppps_selfplagiarism.csv', index=False)
else:
    possiblepairspatpaper.to_parquet(dataint + 'possiblepairs_selfplagiarism.parquet', index=False, compression='snappy')


In [ ]:
possiblepairspatpaper
